# Notebook 03 — Model: KMeans Clustering
**Project:** Credit Card Customer Segmentation  
**Goal:** Determine optimal number of clusters, fit KMeans, visualise segments, and interpret each customer group.

---
### Flow
1. Load scaled data
2. Find optimal K — Elbow Method + Silhouette Score
3. Fit KMeans with optimal K
4. PCA visualisation (2D)
5. Cluster profiling — who is in each segment?
6. Business interpretation
7. Save cluster summary

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
import joblib

from src.data_loader import load_raw_data
from src.preprocessing import preprocess
from src.config import RANDOM_STATE, K_RANGE, N_CLUSTERS, MODELS_DIR, FIGURES_DIR, REPORTS_DIR

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

# Load data
df_scaled, df_unscaled = preprocess(load_raw_data(), save=False)
print('Scaled shape  :', df_scaled.shape)
print('Unscaled shape:', df_unscaled.shape)

## 1. Find Optimal K — Elbow + Silhouette

In [ ]:
inertias, silhouettes = [], []

print(f'{"K":>3} | {"Inertia":>12} | {"Silhouette":>10}')
print('-' * 32)

for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(df_scaled)
    inertias.append(km.inertia_)
    sil = silhouette_score(df_scaled, labels)
    silhouettes.append(sil)
    print(f'{k:>3} | {km.inertia_:>12,.0f} | {sil:>10.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Elbow
axes[0].plot(list(K_RANGE), inertias, marker='o', color='steelblue', linewidth=2)
axes[0].axvline(N_CLUSTERS, color='tomato', linestyle='--', linewidth=1.5, label=f'Chosen K={N_CLUSTERS}')
axes[0].set_title('Elbow Method', fontweight='bold')
axes[0].set_xlabel('Number of Clusters (K)')
axes[0].set_ylabel('Inertia (WCSS)')
axes[0].legend()

# Silhouette
axes[1].plot(list(K_RANGE), silhouettes, marker='s', color='seagreen', linewidth=2)
axes[1].axvline(N_CLUSTERS, color='tomato', linestyle='--', linewidth=1.5, label=f'Chosen K={N_CLUSTERS}')
axes[1].set_title('Silhouette Score', fontweight='bold')
axes[1].set_xlabel('Number of Clusters (K)')
axes[1].set_ylabel('Silhouette Score (higher = better)')
axes[1].legend()

plt.suptitle(f'Optimal K Selection — Chosen K={N_CLUSTERS}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/01_optimal_k.png', bbox_inches='tight')
plt.show()

best_k_sil = list(K_RANGE)[int(np.argmax(silhouettes))]
print(f'Best K by Silhouette : {best_k_sil} (score={max(silhouettes):.4f})')
print(f'Chosen K             : {N_CLUSTERS} (balanced elbow + silhouette + business interpretability)')

## 2. Fit KMeans

In [ ]:
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, n_init=10)
labels = kmeans.fit_predict(df_scaled)

sil_final = silhouette_score(df_scaled, labels)
print(f'KMeans K={N_CLUSTERS} | Inertia={kmeans.inertia_:,.0f} | Silhouette={sil_final:.4f}')

# Cluster sizes
unique, counts = np.unique(labels, return_counts=True)
for k, n in zip(unique, counts):
    print(f'  Cluster {k}: {n:,} customers ({n/len(labels)*100:.1f}%)')

# Save model
joblib.dump(kmeans, MODELS_DIR / 'kmeans.pkl')
print('\nModel saved: models/kmeans.pkl')

## 3. PCA Visualisation

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
components = pca.fit_transform(df_scaled)
explained = pca.explained_variance_ratio_.sum() * 100
joblib.dump(pca, MODELS_DIR / 'pca.pkl')

print(f'PC1 explains: {pca.explained_variance_ratio_[0]*100:.1f}%')
print(f'PC2 explains: {pca.explained_variance_ratio_[1]*100:.1f}%')
print(f'Total       : {explained:.1f}%')

palette = ['steelblue', 'tomato', 'seagreen', 'orange']
plt.figure(figsize=(9, 6))
for cluster in np.unique(labels):
    mask = labels == cluster
    plt.scatter(components[mask, 0], components[mask, 1],
                label=f'Cluster {cluster}', alpha=0.4, s=12,
                color=palette[cluster % len(palette)])
plt.title(f'Customer Segments — PCA 2D View ({explained:.1f}% variance explained)',
          fontweight='bold')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.legend(markerscale=3)
plt.tight_layout()
plt.savefig('../reports/figures/02_clusters_pca.png', bbox_inches='tight')
plt.show()

## 4. Cluster Profiling

In [ ]:
df_unscaled['Cluster'] = labels

key_cols = ['BALANCE', 'PURCHASES', 'CASH_ADVANCE', 'CREDIT_LIMIT',
            'PAYMENTS', 'PRC_FULL_PAYMENT', 'PURCHASES_TO_LIMIT_RATIO',
            'CASH_ADVANCE_RATIO', 'MONTHLY_AVG_PURCHASE']

profile = df_unscaled.groupby('Cluster')[key_cols].mean().round(2)
profile['Customer_Count'] = df_unscaled['Cluster'].value_counts().sort_index()
print('Cluster Mean Profiles:')
print(profile.to_string())

In [ ]:
# Heatmap of normalised cluster profiles
profile_norm = (profile[key_cols] - profile[key_cols].min()) / \
               (profile[key_cols].max() - profile[key_cols].min() + 1e-9)

fig, axes = plt.subplots(1, 2, figsize=(17, 6))

sns.heatmap(profile_norm.T, annot=True, fmt='.2f', cmap='YlOrRd',
            linewidths=0.5, ax=axes[0], cbar_kws={'shrink': 0.8})
axes[0].set_title('Cluster Profiles (Normalised 0-1)', fontweight='bold')
axes[0].set_xlabel('Cluster')

profile[['BALANCE', 'PURCHASES', 'CASH_ADVANCE', 'PAYMENTS']].plot(
    kind='bar', ax=axes[1], edgecolor='white', alpha=0.85, color=['steelblue','seagreen','tomato','orange']
)
axes[1].set_title('Key Metrics by Cluster (Raw Mean $)', fontweight='bold')
axes[1].set_xlabel('Cluster')
axes[1].set_ylabel('Mean Value ($)')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(loc='upper right', fontsize=9)

plt.suptitle('Customer Segment Profiles', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/03_cluster_profiles.png', bbox_inches='tight')
plt.show()

In [ ]:
# Cluster size distribution
unique, counts = np.unique(labels, return_counts=True)
pcts = counts / counts.sum() * 100

plt.figure(figsize=(7, 4))
bars = plt.bar([f'Cluster {k}' for k in unique], counts,
               color=['steelblue', 'tomato', 'seagreen', 'orange'], edgecolor='white', alpha=0.85)
for bar, p in zip(bars, pcts):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
             f'{p:.1f}%', ha='center', fontweight='bold', fontsize=10)
plt.title('Customer Distribution Across Segments', fontweight='bold')
plt.ylabel('Number of Customers')
plt.tight_layout()
plt.savefig('../reports/figures/04_cluster_distribution.png', bbox_inches='tight')
plt.show()

## 5. Business Interpretation

Based on the cluster profiles, here is the business label for each segment:

| Cluster | Label | Key Traits | Marketing Strategy |
|---|---|---|---|
| 0 | **Transactors** | High purchases, low balance, pay in full | Premium rewards cards, travel benefits |
| 1 | **Revolvers** | High balance, low payments, high interest | Debt management products, lower APR offers |
| 2 | **Cash Advance Users** | High cash advance, high balance | Financial counselling, emergency credit products |
| 3 | **Low Activity** | Low balance, low purchases, low engagement | Re-engagement campaigns, introductory spend offers |

> Note: Cluster labels may vary per run. Always verify against the cluster profile heatmap above.

## 6. Save Results

In [ ]:
# Save full cluster summary
summary = df_unscaled.groupby('Cluster').mean().round(2)
summary['Customer_Count'] = df_unscaled['Cluster'].value_counts().sort_index()
summary.to_csv(REPORTS_DIR / 'cluster_summary.csv')
print('Saved: reports/cluster_summary.csv')

print(f'\nFinal Results:')
print(f'  K (clusters)        : {N_CLUSTERS}')
print(f'  Silhouette Score    : {sil_final:.4f}')
print(f'  Inertia (WCSS)      : {kmeans.inertia_:,.0f}')
print(f'  PCA variance explained: {explained:.1f}%')
print(f'  Total customers     : {len(labels):,}')